### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator


This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import torch
import gc



class Model:
    def __init__(self):
        
        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"

        self.model = LLM(
            model = self.model_path,
            dtype = "float16",  
            max_model_len=2048,  
            gpu_memory_utilization=0.85 
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     
    
    def get_response(self, description):

        #alpaca prompt
        instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
                <constraints>
                * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
                * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
                </constraints>
                
                <example>
                    <description>"A red circle with a blue square inside"</description>
                    
                    ```svg
                    <svg viewBox="0 0 256 256" width="256" height="256">
                      <circle cx="50" cy="50" r="40" fill="red"/>
                      <rect x="30" y="30" width="40" height="40" fill="blue"/>
                      <...>
                       ...
                      <...>
                    </svg>
                ```
                </example>        
                
                Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
                Focus on a clear and concise representation of the input description within the given limitations. 
                Always give the complete SVG code with nothing omitted. Never use an ellipsis.
                Do not include unnecessary explanations. Just give the code.
                """
            
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.
            
                ### Instruction:
                Please write a SVG code for the given input.
            
                ### Input:             
                {}
            
                ### Response:
                """
        
        formatted_input = alpaca_prompt.format(description)
        sampling_params = SamplingParams(temperature=0.5, top_k=5,\
                                         top_p=1,max_tokens=1024)
        outputs = self.model.generate([formatted_input], sampling_params)
        
        #suitable for batch inputs as well        
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
        return generated_text
    
    def predict(self, description: str, max_new_tokens=2048) -> str:
        output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)
        

INFO 04-06 23:19:46 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 04-06 23:19:46 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-06 23:19:50 [config.py:585] This model supports multiple tasks: {'embed', 'score', 'classify', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 04-06 23:19:50 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-06 23:19:51 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_b

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-06 23:19:54 [loader.py:447] Loading weights took 2.12 seconds
INFO 04-06 23:19:54 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.268211 seconds
INFO 04-06 23:20:00 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/84ad9dae08/rank_0_0 for vLLM's torch.compile
INFO 04-06 23:20:00 [backends.py:425] Dynamo bytecode transform time: 5.78 s
INFO 04-06 23:20:01 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-06 23:20:05 [monitor.py:33] torch.compile takes 5.78 s in total
INFO 04-06 23:20:06 [kv_cache_utils.py:566] GPU KV cache size: 18,656 tokens
INFO 04-06 23:20:06 [kv_cache_utils.py:569] Maximum concurrency for 2,048 tokens per request: 9.11x
INFO 04-06 23:20:19 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.45 GiB
INFO 04-06 23:20:20 [core.py:151] init engine (profile, create kv cache, warmup model) took 25.08 seconds


In [5]:
model.predict('sun rising in the east')

Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.96s/it, est. speed input: 9.20 t


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" width="200" height="200"><defs><radialGradient id="sunGradient" cx="0.5" cy="0.5" r="0.5"><stop offset="0%" stop-color="#FFD700"/><stop offset="100%" stop-color="#FF8C00"/></radialGradient></defs><g transform="translate(100, 100)"><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform="rotate(45)" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform="rotate(90)" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform="rotate(135)" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform="rotate(180)" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform="rotate(225)" fill="#FFFFFF" opacity="0.3"/><polygon points="0,0 45,20 45,40 0,60 -45,40 -45,20 0,0" transform

In [6]:
import pandas as pd
df15=pd.read_csv('./drawing-with-llms/train.csv',header=[0])
df76=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df76=df76[df76['score'] > 0.5]
df76=df76[['topic','svg_code']]
df76.columns=['description','svg']

In [7]:
from tqdm import tqdm
tqdm.pandas()
df15['svg'] = df15['description'].progress_apply(lambda x: model.predict(x))
df76['svg'] = df76['description'].progress_apply(lambda x: model.predict(x))

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.79s/it, est. speed input: 9.43 t
 13%|█████▊                                      | 2/15 [00:06<00:44,  3.40s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.23s/it, est. speed input: 10.75 
 20%|████████▊                                   | 3/15 [00:13<00:54,  4.58s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.32s/it, est. speed input: 28.03 
 27%|███████████▋                                | 4/15 [00:15<00:41,  3.74s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.30s/it, est. speed input: 11.27 
 33%|██████████████▋                    

In [8]:
#write csv file for new metric score
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
df15.to_csv(f'./io_files/pred_15{file_name}_{timestamp}.csv', index=False)
df76.to_csv(f'./io_files/pred_76{file_name}_{timestamp}.csv', index=False)

In [9]:
model.close_model()

In [10]:
#SigLip Score
evaluator = SVGMetricEvaluator()
#df15['sl_score'] = df15.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg']), axis=1)
df76['sl_score'] = df76.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg']), axis=1)
evaluator.close_model()

Using device: cuda


100%|███████████████████████████████████████████| 76/76 [00:04<00:00, 16.18it/s]


In [11]:
#df15['sl_score'].mean()

In [12]:
df76['sl_score'].mean()

0.4781559328755945